<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_10_%E2%80%94_VEGETATION_FRAGMENTATION_AND_CORE_VEGETATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# SECTION 4.10 — VEGETATION FRAGMENTATION AND CORE VEGETATION
# =============================================================================
#
# Publication-ready statistics for:
#
#   TZPR_NLP_Vegetation_PatchSize_2025.tif
#   TZPR_NLP_Small_Vegetation_Patches_2025.tif
#   TZPR_NLP_Core_Vegetation_2025.tif
#
# Outputs:
#
#   1. Vegetation_Fragmentation_2025_Statistics.csv
#   2. Vegetation_Patch_Size_Distribution_2025.csv
#   3. Small_Vegetation_Patch_Statistics_2025.csv
#   4. Core_Vegetation_Statistics_2025.csv
#   5. Vegetation_Fragmentation_Publication_Table.csv
#   6. Vegetation_Fragmentation_Summary.txt
#
# =============================================================================

import os
import math
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage


# =============================================================================
# USER SETTINGS
# =============================================================================

INPUT_DIR = r"D:\TZPR_NLP\Vegetation_Fragmentation"

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "Publication_Statistics_2025"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------------------------------------------------------------------------
# INPUT RASTERS
# -------------------------------------------------------------------------

PATCH_RASTER = os.path.join(
    INPUT_DIR,
    "TZPR_NLP_Vegetation_PatchSize_2025.tif"
)

SMALL_PATCH_RASTER = os.path.join(
    INPUT_DIR,
    "TZPR_NLP_Small_Vegetation_Patches_2025.tif"
)

CORE_RASTER = os.path.join(
    INPUT_DIR,
    "TZPR_NLP_Core_Vegetation_2025.tif"
)


# =============================================================================
# PATCH-SIZE CLASSES
# =============================================================================

PATCH_CLASSES = [
    ("Very small", 0, 1),
    ("Small", 1, 5),
    ("Medium", 5, 25),
    ("Large", 25, 100),
    ("Very large", 100, np.inf)
]


# =============================================================================
# FUNCTIONS
# =============================================================================

def check_file(path):
    """Check whether input raster exists."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nInput raster not found:\n{path}\n"
        )


def read_binary_raster(path):
    """
    Read raster and convert to binary vegetation mask.

    Any valid non-zero pixel is treated as vegetation.
    """

    check_file(path)

    with rasterio.open(path) as src:

        arr = src.read(1)

        nodata = src.nodata

        if nodata is not None:
            valid = arr != nodata
        else:
            valid = np.ones(arr.shape, dtype=bool)

        vegetation = (
            valid &
            np.isfinite(arr) &
            (arr > 0)
        )

        profile = src.profile.copy()

        transform = src.transform

        crs = src.crs

        height = src.height
        width = src.width

        bounds = src.bounds

    return (
        vegetation,
        profile,
        transform,
        crs,
        height,
        width,
        bounds
    )


# =============================================================================
# GEOGRAPHIC PIXEL AREA
# =============================================================================

def calculate_pixel_area_ha(transform, height, width, crs):
    """
    Calculate pixel area in hectares.

    For EPSG:4326, pixel area varies with latitude.
    Row-wise geographic pixel area is therefore calculated.

    For projected CRS, affine pixel area is used.
    """

    if crs is not None and crs.is_geographic:

        # Longitude width in degrees
        dlon = abs(transform.a)

        # Latitude height in degrees
        dlat = abs(transform.e)

        # Earth radius in metres
        R = 6378137.0

        # Top latitude of raster
        top_lat = transform.f

        row_area_ha = np.zeros(height)

        for row in range(height):

            lat_top = top_lat - row * dlat
            lat_bottom = lat_top - dlat

            phi1 = math.radians(lat_bottom)
            phi2 = math.radians(lat_top)

            area_m2 = (
                R ** 2
                * math.radians(dlon)
                * (
                    math.sin(phi2)
                    - math.sin(phi1)
                )
            )

            row_area_ha[row] = area_m2 / 10000.0

        pixel_area_ha = np.repeat(
            row_area_ha[:, None],
            width,
            axis=1
        )

    else:

        pixel_width = abs(transform.a)
        pixel_height = abs(transform.e)

        area_m2 = pixel_width * pixel_height

        pixel_area_ha = np.full(
            (height, width),
            area_m2 / 10000.0
        )

    return pixel_area_ha


# =============================================================================
# LABEL VEGETATION PATCHES
# =============================================================================

def identify_patches(mask):
    """
    Identify connected vegetation patches using 8-neighbour connectivity.
    """

    structure = np.ones((3, 3), dtype=np.uint8)

    labels, num_patches = ndimage.label(
        mask,
        structure=structure
    )

    return labels, num_patches


# =============================================================================
# PATCH STATISTICS
# =============================================================================

def calculate_patch_statistics(
    mask,
    pixel_area_ha,
    name="Vegetation"
):

    labels, num_patches = identify_patches(mask)

    if num_patches == 0:

        return {
            "Category": name,
            "Number_of_Patches": 0,
            "Total_Area_ha": 0,
            "Total_Area_km2": 0,
            "Mean_Patch_Area_ha": 0,
            "Median_Patch_Area_ha": 0,
            "Minimum_Patch_Area_ha": 0,
            "Maximum_Patch_Area_ha": 0,
            "Std_Patch_Area_ha": 0
        }, np.array([]), labels

    patch_ids = np.arange(
        1,
        num_patches + 1
    )

    # Area of every pixel
    area_values = pixel_area_ha[mask]

    # Corresponding patch label
    label_values = labels[mask]

    # Sum pixel areas by patch
    patch_areas = np.bincount(
        label_values,
        weights=area_values,
        minlength=num_patches + 1
    )[1:]

    patch_areas = patch_areas[
        patch_areas > 0
    ]

    total_area = patch_areas.sum()

    stats = {
        "Category": name,
        "Number_of_Patches": len(patch_areas),
        "Total_Area_ha": total_area,
        "Total_Area_km2": total_area / 100.0,
        "Mean_Patch_Area_ha": np.mean(patch_areas),
        "Median_Patch_Area_ha": np.median(patch_areas),
        "Minimum_Patch_Area_ha": np.min(patch_areas),
        "Maximum_Patch_Area_ha": np.max(patch_areas),
        "Std_Patch_Area_ha": np.std(
            patch_areas,
            ddof=1
        ) if len(patch_areas) > 1 else 0
    }

    return stats, patch_areas, labels


# =============================================================================
# PATCH SIZE DISTRIBUTION
# =============================================================================

def patch_size_distribution(
    patch_areas,
    total_vegetation_area
):

    rows = []

    for class_name, lower, upper in PATCH_CLASSES:

        if upper == np.inf:

            selected = patch_areas >= lower

        else:

            selected = (
                (patch_areas >= lower) &
                (patch_areas < upper)
            )

        areas = patch_areas[selected]

        number = len(areas)

        area = areas.sum()

        if total_vegetation_area > 0:
            percentage = (
                area /
                total_vegetation_area
                * 100
            )
        else:
            percentage = 0

        rows.append({
            "Patch_Size_Class": class_name,
            "Lower_Limit_ha": lower,
            "Upper_Limit_ha":
                upper if upper != np.inf else "No upper limit",
            "Number_of_Patches": number,
            "Area_ha": area,
            "Area_km2": area / 100.0,
            "Percentage_of_Vegetation": percentage
        })

    return pd.DataFrame(rows)


# =============================================================================
# SMALL PATCH STATISTICS
# =============================================================================

def calculate_small_patch_statistics(
    patch_areas,
    total_area
):

    # Small vegetation patches defined as < 5 ha
    small = patch_areas < 5

    small_areas = patch_areas[small]

    number_small = len(small_areas)

    area_small = small_areas.sum()

    if len(patch_areas) > 0:
        percentage_patches = (
            number_small /
            len(patch_areas)
            * 100
        )
    else:
        percentage_patches = 0

    if total_area > 0:
        percentage_area = (
            area_small /
            total_area
            * 100
        )
    else:
        percentage_area = 0

    return {
        "Small_Patch_Definition": "< 5 ha",
        "Number_of_Small_Patches": number_small,
        "Small_Patch_Area_ha": area_small,
        "Small_Patch_Area_km2": area_small / 100.0,
        "Percentage_of_All_Patches": percentage_patches,
        "Percentage_of_Vegetation_Area": percentage_area,
        "Mean_Small_Patch_Area_ha":
            np.mean(small_areas)
            if number_small > 0 else 0,
        "Median_Small_Patch_Area_ha":
            np.median(small_areas)
            if number_small > 0 else 0,
        "Maximum_Small_Patch_Area_ha":
            np.max(small_areas)
            if number_small > 0 else 0
    }


# =============================================================================
# CORE VEGETATION STATISTICS
# =============================================================================

def calculate_core_statistics(
    core_mask,
    pixel_area_ha,
    total_vegetation_area
):

    core_stats, core_patch_areas, _ = (
        calculate_patch_statistics(
            core_mask,
            pixel_area_ha,
            name="Core Vegetation"
        )
    )

    core_area = core_stats[
        "Total_Area_ha"
    ]

    if total_vegetation_area > 0:

        core_percentage = (
            core_area /
            total_vegetation_area
            * 100
        )

    else:

        core_percentage = 0

    result = {
        "Core_Vegetation_Area_ha":
            core_area,

        "Core_Vegetation_Area_km2":
            core_area / 100.0,

        "Core_Vegetation_Percentage":
            core_percentage,

        "Number_of_Core_Patches":
            core_stats["Number_of_Patches"],

        "Mean_Core_Patch_Area_ha":
            core_stats["Mean_Patch_Area_ha"],

        "Median_Core_Patch_Area_ha":
            core_stats["Median_Patch_Area_ha"],

        "Minimum_Core_Patch_Area_ha":
            core_stats["Minimum_Patch_Area_ha"],

        "Maximum_Core_Patch_Area_ha":
            core_stats["Maximum_Patch_Area_ha"],

        "Std_Core_Patch_Area_ha":
            core_stats["Std_Patch_Area_ha"]
    }

    return result, core_patch_areas


# =============================================================================
# MAIN PROCESSING
# =============================================================================

def main():

    print("\n")
    print("=" * 80)
    print("VEGETATION FRAGMENTATION ANALYSIS — 2025")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # READ MAIN VEGETATION PATCH RASTER
    # -------------------------------------------------------------------------

    print("\n[1/5] Reading vegetation patch raster...")

    vegetation_mask, profile, transform, crs, height, width, bounds = (
        read_binary_raster(PATCH_RASTER)
    )

    print(
        f"Raster size: {width} × {height}"
    )

    print(
        f"CRS: {crs}"
    )

    # -------------------------------------------------------------------------
    # PIXEL AREA
    # -------------------------------------------------------------------------

    print("\n[2/5] Calculating pixel areas...")

    pixel_area_ha = calculate_pixel_area_ha(
        transform,
        height,
        width,
        crs
    )

    # -------------------------------------------------------------------------
    # VEGETATION PATCH STATISTICS
    # -------------------------------------------------------------------------

    print("\n[3/5] Calculating vegetation patch statistics...")

    vegetation_stats, patch_areas, labels = (
        calculate_patch_statistics(
            vegetation_mask,
            pixel_area_ha,
            name="All Vegetation"
        )
    )

    total_vegetation_area = vegetation_stats[
        "Total_Area_ha"
    ]

    print(
        f"Vegetation area: "
        f"{total_vegetation_area:,.2f} ha"
    )

    print(
        f"Number of patches: "
        f"{vegetation_stats['Number_of_Patches']:,}"
    )

    # -------------------------------------------------------------------------
    # PATCH SIZE DISTRIBUTION
    # -------------------------------------------------------------------------

    distribution = patch_size_distribution(
        patch_areas,
        total_vegetation_area
    )

    distribution_path = os.path.join(
        OUTPUT_DIR,
        "Vegetation_Patch_Size_Distribution_2025.csv"
    )

    distribution.to_csv(
        distribution_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # SMALL PATCH STATISTICS
    # -------------------------------------------------------------------------

    small_stats = calculate_small_patch_statistics(
        patch_areas,
        total_vegetation_area
    )

    small_df = pd.DataFrame(
        [small_stats]
    )

    small_path = os.path.join(
        OUTPUT_DIR,
        "Small_Vegetation_Patch_Statistics_2025.csv"
    )

    small_df.to_csv(
        small_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # READ CORE VEGETATION RASTER
    # -------------------------------------------------------------------------

    print("\n[4/5] Processing core vegetation...")

    core_mask, _, _, _, _, _, _ = (
        read_binary_raster(
            CORE_RASTER
        )
    )

    core_stats, core_patch_areas = (
        calculate_core_statistics(
            core_mask,
            pixel_area_ha,
            total_vegetation_area
        )
    )

    core_df = pd.DataFrame(
        [core_stats]
    )

    core_path = os.path.join(
        OUTPUT_DIR,
        "Core_Vegetation_Statistics_2025.csv"
    )

    core_df.to_csv(
        core_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # COMBINE PUBLICATION STATISTICS
    # -------------------------------------------------------------------------

    publication = {
        "Year": 2025,

        "Total_Vegetation_Area_ha":
            total_vegetation_area,

        "Total_Vegetation_Area_km2":
            total_vegetation_area / 100.0,

        "Number_of_Vegetation_Patches":
            vegetation_stats["Number_of_Patches"],

        "Mean_Patch_Area_ha":
            vegetation_stats["Mean_Patch_Area_ha"],

        "Median_Patch_Area_ha":
            vegetation_stats["Median_Patch_Area_ha"],

        "Minimum_Patch_Area_ha":
            vegetation_stats["Minimum_Patch_Area_ha"],

        "Maximum_Patch_Area_ha":
            vegetation_stats["Maximum_Patch_Area_ha"],

        "Std_Patch_Area_ha":
            vegetation_stats["Std_Patch_Area_ha"],

        "Small_Patches_lt5ha_Number":
            small_stats["Number_of_Small_Patches"],

        "Small_Patches_lt5ha_Area_ha":
            small_stats["Small_Patch_Area_ha"],

        "Small_Patches_lt5ha_Area_km2":
            small_stats["Small_Patch_Area_km2"],

        "Small_Patches_Percentage_of_All_Patches":
            small_stats["Percentage_of_All_Patches"],

        "Small_Patches_Percentage_of_Vegetation":
            small_stats["Percentage_of_Vegetation_Area"],

        "Core_Vegetation_Area_ha":
            core_stats["Core_Vegetation_Area_ha"],

        "Core_Vegetation_Area_km2":
            core_stats["Core_Vegetation_Area_km2"],

        "Core_Vegetation_Percentage":
            core_stats["Core_Vegetation_Percentage"],

        "Number_of_Core_Patches":
            core_stats["Number_of_Core_Patches"],

        "Mean_Core_Patch_Area_ha":
            core_stats["Mean_Core_Patch_Area_ha"],

        "Median_Core_Patch_Area_ha":
            core_stats["Median_Core_Patch_Area_ha"],

        "Minimum_Core_Patch_Area_ha":
            core_stats["Minimum_Core_Patch_Area_ha"],

        "Maximum_Core_Patch_Area_ha":
            core_stats["Maximum_Core_Patch_Area_ha"]
    }

    publication_df = pd.DataFrame(
        [publication]
    )

    publication_path = os.path.join(
        OUTPUT_DIR,
        "Vegetation_Fragmentation_Publication_Table.csv"
    )

    publication_df.to_csv(
        publication_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # FULL STATISTICS
    # -------------------------------------------------------------------------

    full_stats = pd.DataFrame(
        [
            vegetation_stats
        ]
    )

    full_stats_path = os.path.join(
        OUTPUT_DIR,
        "Vegetation_Fragmentation_2025_Statistics.csv"
    )

    full_stats.to_csv(
        full_stats_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # TEXT SUMMARY
    # -------------------------------------------------------------------------

    summary_path = os.path.join(
        OUTPUT_DIR,
        "Vegetation_Fragmentation_Summary.txt"
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "SECTION 4.10 — VEGETATION FRAGMENTATION "
            "AND CORE VEGETATION\n"
        )

        f.write(
            "=" * 70 + "\n\n"
        )

        f.write(
            "2025 VEGETATION PATCH STRUCTURE\n"
        )

        f.write(
            f"Total vegetation area: "
            f"{total_vegetation_area:,.2f} ha\n"
        )

        f.write(
            f"Total vegetation area: "
            f"{total_vegetation_area / 100.0:,.2f} km²\n"
        )

        f.write(
            f"Number of vegetation patches: "
            f"{vegetation_stats['Number_of_Patches']:,}\n"
        )

        f.write(
            f"Mean patch area: "
            f"{vegetation_stats['Mean_Patch_Area_ha']:,.2f} ha\n"
        )

        f.write(
            f"Median patch area: "
            f"{vegetation_stats['Median_Patch_Area_ha']:,.2f} ha\n"
        )

        f.write(
            f"Minimum patch area: "
            f"{vegetation_stats['Minimum_Patch_Area_ha']:,.4f} ha\n"
        )

        f.write(
            f"Maximum patch area: "
            f"{vegetation_stats['Maximum_Patch_Area_ha']:,.2f} ha\n"
        )

        f.write(
            "\nSMALL VEGETATION PATCHES (<5 ha)\n"
        )

        f.write(
            f"Number: "
            f"{small_stats['Number_of_Small_Patches']:,}\n"
        )

        f.write(
            f"Area: "
            f"{small_stats['Small_Patch_Area_ha']:,.2f} ha\n"
        )

        f.write(
            f"Percentage of all patches: "
            f"{small_stats['Percentage_of_All_Patches']:.2f}%\n"
        )

        f.write(
            f"Percentage of vegetation area: "
            f"{small_stats['Percentage_of_Vegetation_Area']:.2f}%\n"
        )

        f.write(
            "\nCORE VEGETATION\n"
        )

        f.write(
            f"Core area: "
            f"{core_stats['Core_Vegetation_Area_ha']:,.2f} ha\n"
        )

        f.write(
            f"Core area: "
            f"{core_stats['Core_Vegetation_Area_km2']:,.2f} km²\n"
        )

        f.write(
            f"Core vegetation percentage: "
            f"{core_stats['Core_Vegetation_Percentage']:.2f}%\n"
        )

        f.write(
            f"Number of core patches: "
            f"{core_stats['Number_of_Core_Patches']:,}\n"
        )

        f.write(
            f"Mean core patch area: "
            f"{core_stats['Mean_Core_Patch_Area_ha']:,.2f} ha\n"
        )

        f.write(
            f"Median core patch area: "
            f"{core_stats['Median_Core_Patch_Area_ha']:,.2f} ha\n"
        )

        f.write(
            f"Largest core patch: "
            f"{core_stats['Maximum_Core_Patch_Area_ha']:,.2f} ha\n"
        )

    # -------------------------------------------------------------------------
    # PRINT RESULTS
    # -------------------------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("PUBLICATION-READY RESULTS")
    print("=" * 80)

    print(
        f"\nTotal vegetation area: "
        f"{total_vegetation_area:,.2f} ha"
    )

    print(
        f"Vegetation patches: "
        f"{vegetation_stats['Number_of_Patches']:,}"
    )

    print(
        f"Mean patch area: "
        f"{vegetation_stats['Mean_Patch_Area_ha']:,.2f} ha"
    )

    print(
        f"Median patch area: "
        f"{vegetation_stats['Median_Patch_Area_ha']:,.2f} ha"
    )

    print(
        f"Largest patch: "
        f"{vegetation_stats['Maximum_Patch_Area_ha']:,.2f} ha"
    )

    print(
        f"\nSmall patches (<5 ha): "
        f"{small_stats['Number_of_Small_Patches']:,}"
    )

    print(
        f"Small-patch area: "
        f"{small_stats['Small_Patch_Area_ha']:,.2f} ha"
    )

    print(
        f"Small patches (% of all patches): "
        f"{small_stats['Percentage_of_All_Patches']:.2f}%"
    )

    print(
        f"Small patches (% of vegetation): "
        f"{small_stats['Percentage_of_Vegetation_Area']:.2f}%"
    )

    print(
        f"\nCore vegetation: "
        f"{core_stats['Core_Vegetation_Area_ha']:,.2f} ha"
    )

    print(
        f"Core vegetation (%): "
        f"{core_stats['Core_Vegetation_Percentage']:.2f}%"
    )

    print(
        f"Core patches: "
        f"{core_stats['Number_of_Core_Patches']:,}"
    )

    print(
        f"Largest core patch: "
        f"{core_stats['Maximum_Core_Patch_Area_ha']:,.2f} ha"
    )

    print("\n")
    print("=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)

    print(
        f"\n{full_stats_path}"
    )

    print(
        f"{distribution_path}"
    )

    print(
        f"{small_path}"
    )

    print(
        f"{core_path}"
    )

    print(
        f"{publication_path}"
    )

    print(
        f"{summary_path}"
    )

    print("\nAnalysis completed successfully.")


# =============================================================================
# RUN
# =============================================================================

if __name__ == "__main__":
    main()